# PySpark Practical Notes — Quick-Commerce Orders

Hands-on PySpark notebook covering DataFrames, custom schemas, joins, aggregations, date/time transformations, S3 writes, partitioning, `coalesce`, `repartition`, UDFs, and AQE concepts using a Blinkit-style quick-commerce dataset.

> **Environment:** PySpark on EMR with S3 input/output paths.

### Topics covered
- Reading CSV files with explicit schemas
- Shuffle Hash Join
- Sort Merge Join
- Aggregations with `groupBy` and `agg`
- CSV and Parquet writes
- Date/time transformations
- Partitioned writes
- `coalesce()` vs `repartition()`
- Python UDF vs Spark built-in functions
- AQE concepts and limitations


## Dataset Overview

This notebook uses three CSV files from a Blinkit-style quick-commerce dataset:

1. `blinkit_orders.csv` — order header information.
2. `blinkit_order_items.csv` — product line items for each order.
3. `blinkit_products.csv` — product and category attributes.


## 1. Session And Path Setup

Run this notebook on EMR with a PySpark kernel. The only value to change for a new class environment is `BUCKET_NAME`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

BUCKET_NAME = "spark-file-storage"

base_path = f"s3://{BUCKET_NAME}/spark-learning"
raw_path = f"{base_path}/blinkit/raw"
output_path = f"{base_path}/blinkit/output"

orders_csv_path = f"{raw_path}/blinkit_orders.csv"
order_items_csv_path = f"{raw_path}/blinkit_order_items.csv"
products_csv_path = f"{raw_path}/blinkit_products.csv"

## 2. Read CSV With Custom Schema

For class consistency, the rest of the notebook uses explicit schemas. If your CSV has extra columns, Spark ignores the extras only when the schema and file are compatible by position; keep the CSV header aligned with these columns for a smooth run.

In [ ]:
# Custom schema for the order header file.
# The schema below matches the sample CSV header exactly:
# order_id, customer_id, order_date, promised_delivery_time, actual_delivery_time,
# delivery_status, order_total, payment_method, delivery_partner_id, store_id.
# Keep timestamp fields as strings during the read, then parse them explicitly in transformation cells.
orders_schema = T.StructType([
    T.StructField("order_id", T.LongType(), False),
    T.StructField("customer_id", T.LongType(), True),
    T.StructField("order_date", T.StringType(), True),
    T.StructField("promised_delivery_time", T.StringType(), True),
    T.StructField("actual_delivery_time", T.StringType(), True),
    T.StructField("delivery_status", T.StringType(), True),
    T.StructField("order_total", T.DoubleType(), True),
    T.StructField("payment_method", T.StringType(), True),
    T.StructField("delivery_partner_id", T.LongType(), True),
    T.StructField("store_id", T.LongType(), True),
])

orders_df = (
    spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .option("escape", '"')
    .schema(orders_schema)
    .csv(orders_csv_path)
)

orders_df.printSchema()
orders_df.show(5, truncate=False)

## 3. Read The Two Supporting CSV Files

These two DataFrames are enough for realistic joins and product-level analysis.

In [ ]:
order_items_schema = T.StructType([
    T.StructField("order_id", T.LongType(), False),
    T.StructField("product_id", T.LongType(), False),
    T.StructField("quantity", T.IntegerType(), True),
    T.StructField("unit_price", T.DoubleType(), True),
])

products_schema = T.StructType([
    T.StructField("product_id", T.LongType(), False),
    T.StructField("product_name", T.StringType(), True),
    T.StructField("category", T.StringType(), True),
    T.StructField("brand", T.StringType(), True),
    T.StructField("price", T.DoubleType(), True),
    T.StructField("mrp", T.DoubleType(), True),
    T.StructField("margin_percentage", T.DoubleType(), True),
    T.StructField("shelf_life_days", T.IntegerType(), True),
    T.StructField("min_stock_level", T.IntegerType(), True),
    T.StructField("max_stock_level", T.IntegerType(), True),
])

order_items_df = (
    spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .option("escape", '"')
    .schema(order_items_schema)
    .csv(order_items_csv_path)
)

products_df = (
    spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .option("escape", '"')
    .schema(products_schema)
    .csv(products_csv_path)
)

order_items_df.show(5, truncate=False)
products_df.show(5, truncate=False)

## 4. Shuffle Hash Join

Shuffle hash join redistributes both sides by the join key and builds hash tables after the shuffle. Spark may still choose a different physical plan if the data or settings make another strategy better.

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.sql.join.preferSortMergeJoin", "false")
# spark.conf.set("spark.sql.shuffle.partitions", "8")

# spark.sql.adaptive.enabled = true by default
# spark.sql.shuffle.partitions -  this is the property which controls number of shuffle partitions
# default value is 200

shuffle_hash_join_df = (
    order_items_df.alias("oi").hint("SHUFFLE_HASH")
    .join(products_df.alias("p").hint("SHUFFLE_HASH"), on=F.col("oi.product_id") == F.col("p.product_id"), how="inner")
    .select(
        F.col("oi.order_id"),
        F.col("oi.product_id"),
        F.col("p.product_name"),
        F.col("p.category"),
        F.col("oi.quantity"),
        F.col("oi.unit_price")
    )
)

spark.conf.set("spark.sql.adaptive.enabled", "false")
shuffle_hash_join_df.explain("formatted")
shuffle_hash_join_df.show(10, truncate=False)
print("Number of partitions:", shuffle_hash_join_df.rdd.getNumPartitions())

# 110 parts got created - each was having 1 mb of data, total data 110mb 
# 1 partition = 1 task = 1 cpu core
# logically there will be 110 tasks , each task will process 1 mb of data
# maxPartitionBytesize - default value of this is 128mb


# What AQE is
# Adaptive Query Execution (spark.sql.adaptive.enabled, on by default since Spark 3.2) lets Spark re-optimize a query at runtime using real statistics instead of committing to a plan built purely from compile-time estimates. After each shuffle stage completes, Spark looks at the actual data sizes and can revise the rest of the plan. Its three main tricks:

# Dynamically coalescing shuffle partitions — collapses the static 200 into fewer, right-sized partitions based on real output size (via advisoryPartitionSizeInBytes).
# Switching join strategy at runtime — if a side turns out smaller than expected after filtering, it demotes a planned sort-merge join to a broadcast join.
# Skew join handling — detects an oversized (skewed) partition and splits it into smaller sub-partitions so one task doesn't become a straggler.

# Cases where AQE doesn't help (the "when is it useless" part)

# The unifying theme: AQE only acts at shuffle boundaries. No shuffle → nothing for it to re-optimize.
# Queries with no shuffle at all. A plain read → filter → select → write, or a job that's already a broadcast join, has no exchange stage — AQE has no decision point and does nothing.
# Very small datasets. Coalescing 200 partitions down helps big jobs; on tiny data the overhead saved is negligible, and you'd have set partitions low manually anyway.
# When you've already hard-coded the right plan. If you forced broadcast() or manually set shuffle.partitions correctly, AQE has nothing better to offer — it might even be redundant with your hints.
# Compile-time-only problems. AQE can't fix a bad logical plan, a missing filter, an exploding join, or poor file layout — it only adjusts the physical plan after stages run. It's not a substitute for good query design or partitioning.


# AQE will combine (coalesce) it into 1 partition of total 110 mb size, 1 cpu core and it will be 1 task
# spark.sql.adaptive.coalescePartitions.enabled = true
# spark.sql.adaptive.coalescePartitions.minPartitionSize = default is 1 mb, partitions won't be lower than this
# spark.sql.adaptive.advisoryPartitionSizeInBytes - default 128 mb

## 5. Sort Merge Join

Sort merge join is common for large joins. Spark shuffles both sides by the join key, sorts matching partitions, and then merges matching rows.

In [ ]:
# Disable broadcast so the physical plan can show a large-table join strategy.
# Sort merge join is usually preferred for large equi-joins when broadcast is not suitable.
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.sql.join.preferSortMergeJoin", "true")
spark.conf.set("spark.sql.shuffle.partitions", "8")

sort_merge_join_df = (
    orders_df.alias("o").hint("MERGE")
    .join(
        order_items_df.alias("oi").hint("MERGE"),
        on=F.col("o.order_id") == F.col("oi.order_id"),
        how="inner"
    )
    .select(
        F.col("o.order_id"),
        F.col("o.store_id"),
        F.col("o.customer_id"),
        F.col("o.delivery_status"),
        F.col("oi.product_id"),
        F.col("oi.quantity"),
        F.col("oi.unit_price"),
        (F.col("oi.quantity") * F.col("oi.unit_price")).alias("line_amount")
    )
)

sort_merge_join_df.explain("formatted")
sort_merge_join_df.show(10, truncate=False)
print("Number of partitions:", sort_merge_join_df.rdd.getNumPartitions())


In [ ]:
store_order_summary_df = (
    orders_df
    .groupBy(F.col("store_id"))
    .agg(
        F.countDistinct(F.col("order_id")).alias("total_orders"),
        F.countDistinct(F.col("customer_id")).alias("unique_customers"),
        F.round(F.sum(F.col("order_total")), 2).alias("total_revenue"),
        F.round(F.avg(F.col("order_total")), 2).alias("avg_order_value"),
        F.max(F.col("order_total")).alias("highest_order_value")
    )
    .orderBy(F.col("total_revenue").desc_nulls_last())
)

store_order_summary_df.show(20, truncate=False)

## 6. Write CSV Data To S3

CSV is easy to inspect and share, but it is not the best storage format for repeated analytics.

In [ ]:
csv_output_path = f"{output_path}/store_order_summary_csv"

(
    store_order_summary_df
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(csv_output_path)
)

print("CSV written to:", csv_output_path)

## 7. Write Parquet Data To S3

Parquet stores schema and columnar data, so Spark can read only the columns needed by a query.

In [ ]:
parquet_output_path = f"{output_path}/store_order_summary_parquet"

(
    store_order_summary_df
    .write
    .mode("overwrite")
    .parquet(parquet_output_path)
)

print("Parquet written to:", parquet_output_path)

In [ ]:
amount_col = F.coalesce(F.col("order_total"), F.lit(0.0))
timestamp_pattern = "yyyy-MM-dd HH:mm:ss"

orders_with_date_df = (
    orders_df
    .withColumn("order_ts", F.to_timestamp(F.col("order_date"), timestamp_pattern))
    .withColumn("promised_delivery_ts", F.to_timestamp(F.col("promised_delivery_time"), timestamp_pattern))
    .withColumn("actual_delivery_ts", F.to_timestamp(F.col("actual_delivery_time"), timestamp_pattern))
    .withColumn("order_dt", F.to_date(F.col("order_ts")))
    .withColumn("order_month", F.date_format(F.col("order_dt"), "yyyy-MM"))
    .withColumn("order_hour", F.hour(F.col("order_ts")))
    .withColumn(
        "delivery_delay_minutes",
        F.round((F.col("actual_delivery_ts").cast("long") - F.col("promised_delivery_ts").cast("long")) / 60, 2)
    )
    .withColumn("platform_fee", F.round(amount_col * F.lit(0.02), 2))
    .withColumn("estimated_seller_payout", F.round(amount_col - F.col("platform_fee"), 2))
)

orders_with_date_df.show(5)

## 8. Partitioned Write

Partitioning writes files into folder structures based on column values. Choose low- or medium-cardinality columns that are frequently used in filters.

In [ ]:
# Partition by order date for date-based reporting.
# Avoid partitioning by very high-cardinality columns such as order_id; that creates many tiny folders.
orders_partition_ready_df = (
    orders_with_date_df
    .filter(F.col("order_dt").isNotNull())
    .select(
        "order_id",
        "customer_id",
        "store_id",
        "delivery_status",
        "order_total",
        "order_month",
        "order_dt"
    )
)

partitioned_output_path = f"{output_path}/orders_partitioned_by_date"

(
    orders_partition_ready_df
    .write
    .mode("overwrite")
    .partitionBy("order_dt")
    .parquet(partitioned_output_path)
)

# We can control row group sizes while writing data in parquet form

# parquet_tuning_path = f"{output_path}/orders_parquet_tuned"

# (
#     orders_with_date_df
#     .write
#     .mode("overwrite")
#     .option("compression", "snappy")
#     .option("maxRecordsPerFile", 100000)              # max records as output for each parquet file
#     .option("parquet.block.size", 64 * 1024 * 1024)   # 64 MB row group target
#     .option("parquet.page.size", 1 * 1024 * 1024)      # 1 MB page target
#     .parquet(parquet_tuning_path)
# )

print("Partitioned Parquet written to:", partitioned_output_path)

## 9. Coalesce Write

`coalesce` reduces the number of partitions without a full shuffle. It is useful when writing a small result for human inspection.

In [ ]:
# coalesce(1) creates one output data file, which is convenient for a small class demo.
# Do not use coalesce(1) for large production outputs because one executor must write the final file.
single_file_csv_path = f"{output_path}/store_order_summary_single_csv"

(
    store_order_summary_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(single_file_csv_path)
)

print("Single-partition CSV folder written to:", single_file_csv_path)

## 10. Repartition Write

`repartition` changes the number of partitions using a shuffle. Use it when you need better parallelism or want data distributed by a specific column before writing.

In [ ]:
# Repartition by store_id before writing store-based output.
# This causes a shuffle, so use it when the downstream benefit is worth the movement of data.
repartitioned_store_path = f"{output_path}/orders_repartitioned_by_store"

(
    orders_df
    .repartition("store_id")
    .write
    .mode("overwrite")
    .partitionBy("store_id")
    .parquet(repartitioned_store_path)
)

print("Store-partitioned Parquet written to:", repartitioned_store_path)

# repartitioned_store_path = f"{output_path}/orders_repartitioned_by_store"

# (
#     orders_df
#     .repartition(2, "store_id")
#     .write
#     .mode("overwrite")
#     .parquet(repartitioned_store_path)
# )

# print("Store-partitioned Parquet written to:", repartitioned_store_path)

## 11. UDF In Spark

UDFs let you run custom Python logic on Spark rows. Use Spark built-in functions first when possible because they are easier for Spark to optimize.

In [ ]:
# A small UDF that creates a stock-policy label from min and max stock levels.
# The source file does not contain live stock; it contains planning thresholds.
@F.udf(returnType=T.StringType())
def stock_policy(min_stock_level, max_stock_level):
    if min_stock_level is None or max_stock_level is None:
        return "Unknown"
    stock_range = max_stock_level - min_stock_level
    if stock_range < 30:
        return "Tight Range"
    if stock_range < 60:
        return "Standard Range"
    return "Wide Range"

products_stock_policy_df = products_df.withColumn(
    "stock_policy",
    stock_policy(F.col("min_stock_level"), F.col("max_stock_level"))
)

products_stock_policy_df.select(
    F.col("product_id"),
    F.col("product_name"),
    F.col("category"),
    F.col("min_stock_level"),
    F.col("max_stock_level"),
    F.col("stock_policy")
).show(20, truncate=False)

## 12. UDF Alternative With Built-In Functions

The same logic can be expressed with `when`. For simple column logic like this, prefer the built-in version.

In [ ]:
products_stock_policy_builtin_df = products_df.withColumn(
    "stock_policy",
    F.when(
        F.col("min_stock_level").isNull() | F.col("max_stock_level").isNull(),
        "Unknown"
    ).when(
        (F.col("max_stock_level") - F.col("min_stock_level")) < 30,
        "Tight Range"
    ).when(
        (F.col("max_stock_level") - F.col("min_stock_level")) < 60,
        "Standard Range"
    ).otherwise("Wide Range")
)

products_stock_policy_builtin_df.select(
    F.col("product_id"),
    F.col("product_name"),
    F.col("min_stock_level"),
    F.col("max_stock_level"),
    F.col("stock_policy")
).show(20, truncate=False)